# Ollama local smoke test (macOS)

Este notebook te ayuda a confirmar rápidamente si **Ollama está corriendo en local** y si **un modelo responde**.

- Base URL por defecto: `http://localhost:11434`
- Endpoints útiles:
  - API Ollama: `GET /api/tags`, `POST /api/generate`
  - OpenAI-compatible: `POST /v1/chat/completions`

Si el servicio no está levantado, en otra terminal:

```bash
ollama serve
```


In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from typing import Any

import httpx

BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
MODEL = os.environ.get("OLLAMA_MODEL", "llama3.2:3b")

print("BASE_URL:", BASE_URL)
print("MODEL:", MODEL)

ollama_path = shutil.which("ollama")
print("ollama CLI:", ollama_path or "NOT FOUND")

def get_json(path: str) -> dict[str, Any]:
    url = f"{BASE_URL}{path}"
    with httpx.Client(timeout=3.0) as client:
        r = client.get(url)
        r.raise_for_status()
        return r.json()

ok_tags = False
try:
    tags = get_json("/api/tags")
    ok_tags = True
    models = [m.get("name") for m in tags.get("models", [])]
    print("/api/tags OK. models:")
    for m in models:
        print("-", m)
except Exception as e:  # noqa: BLE001
    print("/api/tags FAILED:", repr(e))

ok_openai = False
try:
    # Ollama expone OpenAI-compatible; /v1/models suele responder cuando está arriba.
    openai_models = get_json("/v1/models")
    ok_openai = True
    print("/v1/models OK. first entries:")
    for item in (openai_models.get("data") or [])[:5]:
        print("-", item.get("id"))
except Exception as e:  # noqa: BLE001
    print("/v1/models FAILED:", repr(e))

print("\nRESULT:")
print("- Ollama API reachable:", ok_tags)
print("- OpenAI-compatible reachable:", ok_openai)


BASE_URL: http://localhost:11434
MODEL: llama3.2:3b
ollama CLI: /opt/homebrew/bin/ollama
/api/tags OK. models:
- llama3.2:3b
- deepseek-r1:8b
/v1/models OK. first entries:
- llama3.2:3b
- deepseek-r1:8b

RESULT:
- Ollama API reachable: True
- OpenAI-compatible reachable: True


In [2]:
# (Opcional) Ver modelos vía CLI, si tienes el binario.
if ollama_path:
    r = subprocess.run(["ollama", "list"], text=True, capture_output=True)
    print("exit:", r.returncode)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print("Skipping: ollama CLI not found in PATH")


exit: 0
NAME              ID              SIZE      MODIFIED       
llama3.2:3b       a80c4f17acd5    2.0 GB    19 minutes ago    
deepseek-r1:8b    6995872bfe4c    5.2 GB    12 days ago


In [3]:
# Prueba de generación mínima vía API Ollama nativa.
prompt = "Say 'pong' and nothing else."
payload = {
    "model": MODEL,
    "prompt": prompt,
    "stream": False,
}

url = f"{BASE_URL}/api/generate"
with httpx.Client(timeout=60.0) as client:
    r = client.post(url, json=payload)
    print("status:", r.status_code)
    r.raise_for_status()
    data = r.json()

print("response:", (data.get("response") or "").strip())
print("done:", data.get("done"))


status: 200
response: pong
done: True


In [4]:
# Prueba vía endpoint OpenAI-compatible (chat.completions)
# Nota: la "API key" puede ser cualquier string; Ollama la ignora.
payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Reply with exactly: pong"},
    ],
    "temperature": 0,
}

url = f"{BASE_URL}/v1/chat/completions"
headers = {"Authorization": "Bearer ollama"}
with httpx.Client(timeout=60.0) as client:
    r = client.post(url, json=payload, headers=headers)
    print("status:", r.status_code)
    r.raise_for_status()
    data = r.json()

msg = (((data.get("choices") or [{}])[0]).get("message") or {}).get("content")
print("assistant:", (msg or "").strip())


status: 200
assistant: Pong


## Pruebas rápidas (2 prompts)

Edita `PROMPT_1` y `PROMPT_2` y corre la celda de abajo.


In [5]:
PROMPT_1 = "Dime en una palabra: pong"
PROMPT_2 = "Resume en 1 frase qué es Ollama."


def ollama_generate(prompt: str) -> str:
    url = f"{BASE_URL}/api/generate"
    payload = {"model": MODEL, "prompt": prompt, "stream": False}
    with httpx.Client(timeout=60.0) as client:
        r = client.post(url, json=payload)
        r.raise_for_status()
        data = r.json()
    return (data.get("response") or "").strip()


def openai_chat(prompt: str) -> str:
    url = f"{BASE_URL}/v1/chat/completions"
    headers = {"Authorization": "Bearer ollama"}
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are concise."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0,
    }
    with httpx.Client(timeout=60.0) as client:
        r = client.post(url, json=payload, headers=headers)
        r.raise_for_status()
        data = r.json()
    msg = (((data.get("choices") or [{}])[0]).get("message") or {}).get("content")
    return (msg or "").strip()


print("=== Native /api/generate ===")
print("PROMPT_1 ->", ollama_generate(PROMPT_1))
print("PROMPT_2 ->", ollama_generate(PROMPT_2))

print("\n=== OpenAI /v1/chat/completions ===")
print("PROMPT_1 ->", openai_chat(PROMPT_1))
print("PROMPT_2 ->", openai_chat(PROMPT_2))


=== Native /api/generate ===
PROMPT_1 -> Pong
PROMPT_2 -> Ollama es una plataforma de inteligencia artificial y aprendizaje automático que permite a los usuarios crear y entrenar modelos de lenguaje para responder a preguntas y realizar tareas específicas de manera automatizada.

=== OpenAI /v1/chat/completions ===
PROMPT_1 -> Punto.
PROMPT_2 -> Ollama es una plataforma de aprendizaje en línea que utiliza inteligencia artificial para ofrecer cursos personalizados y adaptados a las necesidades de cada estudiante.


## Siguiente paso: correr el loop del agente contra Ollama

En esta misma carpeta ya existe un runner que levanta (si hace falta) el tool server y ejecuta el loop del agente contra Ollama:

```bash
cd verl-tool
python3 scripts/local-llm-ollama/run_test_agent_with_ollama.py \
  --ollama-base-url http://localhost:11434/v1 \
  --ollama-model llama3.2:3b \
  --tool-port 5500 \
  --dump trajectory_ollama.json
```

Tips:
- Si no tienes el modelo: `ollama pull llama3.2:3b`
- Lista de modelos: `ollama list`
